<a href="https://colab.research.google.com/github/anmollate/Data-Science/blob/main/ColumnTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/Datasets/covid_dataset.csv")

In [ ]:
df.head(5)

,age,gender,fever,cough,city,has_covid
0,52,Male,101.3,mild,Bangalore,1
1,15,Male,98.9,mild,Bangalore,1
2,72,Female,NaN,mild,Mumbai,1
3,61,Male,99.3,strong,Delhi,1
4,21,Female,101.8,strong,Bangalore,0


In [ ]:
#getting the ditribution of population in the cities
df['city'].value_counts()

,count
city,
Mumbai,87
Bangalore,76
Delhi,72
Chennai,65


In [ ]:
#same for gender
df['gender'].value_counts()

,count
gender,
Female,160
Male,140


In [ ]:
#getting the null values in each column
df.isnull().sum()

,0
age,0
gender,0
fever,30
cough,0
city,0
has_covid,0


In [ ]:
#Preprocessing to be done
# gender --> OneHotEncoding since gender has no order
# fever --> SimpleImputer here simple imputer because the fever column has some null values
# cough --> OrdinalEncoder since cough has an order where strong cough is more severe than mild cough
# city --> OneHotEncoder also no city is greater than other one. i.e no relationship between the cities

In [ ]:
#Using Normal Approach
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(drop='first',sparse_output=False)
encoder.fit(df[["gender","city"]])
saved_columns=encoder.get_feature_names_out()

df_gender_city=encoder.transform(df[['gender','city']])

In [ ]:
df_gender_city.shape

(300, 4)

In [ ]:
df_gender_city

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 0., 1.],
       [1., 1., 0., 0.],
       [1., 0., 0., 0.]])

In [ ]:
df_gender_city_DF=pd.DataFrame(df_gender_city,columns=saved_columns)

In [ ]:
df_gender_city_DF.head()

,gender_Male,city_Chennai,city_Delhi,city_Mumbai
0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0
3,1.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0


In [ ]:
# Now Imputing The Feature Column Using SimpleImputer
from sklearn.impute import SimpleImputer
imputer=SimpleImputer()
imputer.fit(df[['fever']])

fever_arr=imputer.transform(df[['fever']])

In [ ]:
fever_arr.shape

(300, 1)

In [ ]:
#Converting the fever array into DataFrame
fever_df=pd.DataFrame(fever_arr,columns=["fever"])

In [ ]:
fever_df.head(5)

,fever
0,101.300000
1,98.900000
2,100.307778
3,99.300000
4,101.800000


In [ ]:
#Now Applying Ordinal Encoder On Cough Column
from sklearn.preprocessing import OrdinalEncoder
encoder2=OrdinalEncoder(categories=[["mild","strong"]])

encoder2.fit(df[['cough']])

cough_arr=encoder2.transform(df[['cough']])

In [ ]:
cough_arr.shape

(300, 1)

In [ ]:
#now converting the cough array into DataFrame
cough_df=pd.DataFrame(cough_arr,columns=['cough'])

In [ ]:
cough_df.head(5)

,cough
0,0.0
1,0.0
2,0.0
3,1.0
4,1.0


In [ ]:
featured_data=pd.concat([df['age'],df_gender_city_DF,fever_df,cough_df],axis=1)

In [ ]:
featured_data.head(5)

,age,gender_Male,city_Chennai,city_Delhi,city_Mumbai,fever,cough
0,52,1.0,0.0,0.0,0.0,101.300000,0.0
1,15,1.0,0.0,0.0,0.0,98.900000,0.0
2,72,0.0,0.0,0.0,1.0,100.307778,0.0
3,61,1.0,0.0,1.0,0.0,99.300000,1.0
4,21,0.0,0.0,0.0,0.0,101.800000,1.0


# Now Using ColumnTransformer

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer=ColumnTransformer(
    transformers=[
        ('OHE',OneHotEncoder(sparse_output=False,drop='first'),['gender','city']),
        ('OE',OrdinalEncoder(categories=[['mild','strong']]),['cough']),
        ('SI',SimpleImputer(),['fever'])
    ],remainder='passthrough' #here their are 2 options either let the transform column be dropped or let them passthrough
)

In [ ]:
transformer.fit(df)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


ColumnTransformer(remainder='passthrough',
                  transformers=[('OHE',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city']),
                                ('OE',
                                 OrdinalEncoder(categories=[['mild',
                                                             'strong']]),
                                 ['cough']),
                                ('SI', SimpleImputer(), ['fever'])])

In [ ]:
saved_columns=transformer.get_feature_names_out()

In [ ]:
updated_arr=transformer.transform(df)

In [ ]:
updated_arr.shape

(300, 8)

In [ ]:
updated_df=pd.DataFrame(updated_arr,columns=saved_columns)

In [ ]:
updated_df.head(5)

,OHE__gender_Male,OHE__city_Chennai,OHE__city_Delhi,OHE__city_Mumbai,OE__cough,SI__fever,remainder__age,remainder__has_covid
0,1.0,0.0,0.0,0.0,0.0,101.300000,52.0,1.0
1,1.0,0.0,0.0,0.0,0.0,98.900000,15.0,1.0
2,0.0,0.0,0.0,1.0,0.0,100.307778,72.0,1.0
3,1.0,0.0,1.0,0.0,1.0,99.300000,61.0,1.0
4,0.0,0.0,0.0,0.0,1.0,101.800000,21.0,0.0
